# 第 5 集 — `adsr~`:envelope

⏱ 约 25 分钟 · 难度 ★★☆

**这一集控制的音乐元素:音量随时间的形状(articulation)** — 一个音是 "弹" 出来的还是 "推" 出来的。

**Max 里的对应 patch:** `adsr~` → `*~`。

**你会听到:**
1. 同一个音色、同一个音高,三种 "手感":pluck、pad、organ
2. 没有 envelope 时的 "咔" 一声,以及它是怎么被消掉的
3. 一段 8 个音的 bassline

**学完你能做到:**
- 解释 envelope 为什么 "也只是一个 array"
- 用 `np.interp` 连点成线
- 写出并使用 `adsr()`
- 知道 click 从哪来、怎么去掉
- 把多个带 envelope 的音接成一段 sequence

## 工具箱

前几集做好的东西,原样搬过来。运行一下就行,不用重读(难的地方都有 `#` 注释)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip 把超过 ±1 的部分压住,防止爆音
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # 表里每个位置的 phase,0 到 1
    table = np.zeros(N)                        # 先做一张全是 0 的空表
    for k, amp in enumerate(harmonic_amps, start=1):
        # 第 k 个 harmonic:在表里走 k 个周期,音量 amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize:峰值变成 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # 第 f 张表有几个 harmonic:从 1 个均匀增加到 max_harmonics 个
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # 叠成二维 array:F 行 × N 列

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- 在哪两张表之间 ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # 0..1 换算成 0..F-1 的 "表编号",可以带小数
    f0 = np.floor(fpos).astype(int)            # 下面那张表
    f1 = np.minimum(f0 + 1, F - 1)             # 上面那张表(不超过最后一张)
    ffrac = fpos - f0                          # 离下面那张有多远,0..1

    # --- 在表里的哪两个位置之间(和第 3 集一样)---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # 左边的位置
    i1 = (i0 + 1) % N                          # 右边的位置,到表尾绕回 0
    frac = p - np.floor(p)                     # 小数部分

    # 先在每张表内部插值,再在两张表之间插值
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

frames = make_frames()

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr
phase = (freq * t) % 1

osc = osc_wavetable(frames, phase, 0.6)

`osc` 是一个 2 秒的、音量不变的持续音(wavetable position 0.6)。这一集所有的例子都是在它上面 "盖" 不同的 envelope。

## 1. 最简单的 envelope:fade out

envelope 就是 **一个和声音一样长的 array,值在 0 到 1 之间**。把它和声音逐个 sample 相乘(`*~`),声音在每一刻的音量就被它决定了。

最简单的一条:从 1 直线走到 0。上一集的 `np.linspace` 正好能做。

**应该听到:** 一个音从满音量开始,2 秒内均匀地淡出到无声。

In [ ]:
fade_out = np.linspace(1, 0, len(t))

plt.figure(figsize=(9, 2.5))
plt.plot(t, osc * fade_out, linewidth=0.3)     # 声音 × envelope
plt.plot(t, fade_out, color="red")             # envelope 本身
plt.xlabel("time (s)")
plt.show()

play(osc * fade_out)

红线是 envelope;蓝色是乘完以后的声音,它的轮廓跟着红线走。"envelope"(包络)这个词就是这个意思。

## 2. 连点成线:`np.interp`

直线不够用。我们想要的是 "几段直线连起来" 的形状。

`np.interp(t, 时间点, 高度)`:给几个 (时间, 高度) 的点,它在相邻的点之间画直线,并算出 `t` 里每一刻的高度。相当于 Max 的 `function` object 接 `line~`。

下面三个点:(0 秒, 0) → (0.1 秒, 1) → (1.5 秒, 0)。也就是 0.1 秒升到顶,然后 1.4 秒降到 0。

最后一个点之后,`np.interp` 会 **一直停在最后那个高度**(这里是 0)。

**应该听到:** 一个快速出现、然后慢慢消失的音,1.5 秒后完全无声。

In [ ]:
points_time = [0.0, 0.1, 1.5]
points_level = [0.0, 1.0, 0.0]

env = np.interp(t, points_time, points_level)

plt.figure(figsize=(9, 2.5))
plt.plot(t, env)
plt.plot(points_time, points_level, "o", color="red")     # 我们给的三个点
plt.xlabel("time (s)")
plt.ylabel("level")
plt.show()

play(osc * env)

## 3. Click 从哪来

为什么需要 envelope?除了表现力,还有一个很实际的原因。

下面把一个音在 0.5037 秒处 **直接砍断**(后面接上一段静音)。砍断的那一刻波形不在 0 上,而是突然从某个值跳到 0。

**应该听到:** 音结束的瞬间有一声 "咔"。

**应该看到:** 波形在中间突然被切掉,有一条竖直的跳变。

In [ ]:
n_cut = int(sr * 0.5037)                       # 在第几个 sample 砍断
silence = np.zeros(int(sr * 0.5))              # 0.5 秒的静音 = 一串 0

chopped = np.concatenate([osc[:n_cut], silence])

plt.figure(figsize=(9, 2.5))
plt.plot(chopped[n_cut - 600 : n_cut + 300])   # 只画砍断点附近
plt.title("abrupt cut")
plt.show()

play(chopped)

解决办法:在结尾加一个很短的 fade(这里是 10 毫秒)。10 毫秒短到听不出 "淡出",但足够让波形平滑地回到 0。

**应该听到:** 同样长度的音,结尾干净,没有 "咔"。

这就是 release 最基本的作用。attack 同理:它防止音头的 click。

In [ ]:
t_cut = np.arange(n_cut) / sr
end = n_cut / sr                                           # 音结束的时间(秒)

# 一直是 1,最后 10 ms 降到 0
safe_env = np.interp(t_cut, [0, end - 0.01, end], [1, 1, 0])

faded = np.concatenate([osc[:n_cut] * safe_env, silence])

plt.figure(figsize=(9, 2.5))
plt.plot(faded[n_cut - 600 : n_cut + 300])
plt.title("10 ms fade")
plt.show()

play(faded)

## 4. ADSR

Max 里你按下键、松开键。Python 里没有键盘,所以直接说 "按了多少秒",这个数叫 **gate**。`gate = 1.2` 就是按住 1.2 秒后松开。

五个数字:

| | 名字 | 单位 | 意思 |
|---|---|---|---|
| `a` | attack | 秒 | 从 0 升到 1 用多久 |
| `d` | decay | 秒 | 从 1 降到 sustain 用多久 |
| `s` | sustain | **高度**(0–1) | 按住时停留的高度。**不是时间。** |
| `r` | release | 秒 | 松开后降到 0 用多久 |
| `gate` | | 秒 | 按住多久 |

### 分两段做

**按住期间:** 三个点 (0, 0) → (a, 1) → (a+d, s)。之后 `np.interp` 自动停在 `s`。这正好就是 sustain。

In [ ]:
a, d, s, r, gate = 0.2, 0.3, 0.5, 0.5, 1.2

held = np.interp(t, [0, a, a + d], [0, 1, s])

plt.figure(figsize=(9, 2.5))
plt.plot(t, held)
plt.axvline(gate, linestyle="--", color="gray")
plt.title("held: what the envelope does if the key is never released")
plt.xlabel("time (s)")
plt.show()

**松开之后:** 从松开那一刻的高度,用 `r` 秒直线降到 0。

- `(t - gate) / r`:松开后过了多久,以 `r` 为单位。刚松开是 0,过了 `r` 秒是 1。
- `1 - ...`:倒过来,刚松开是 1,过了 `r` 秒是 0。
- `np.clip(..., 0, 1)`:松开之前这个数会大于 1,`r` 秒之后会小于 0,都压回 0 到 1 之间。
- 再乘上 `level_at_release`(松开那一刻的高度)。

In [ ]:
level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])      # 只问一个时间点:松开时有多高
released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)

plt.figure(figsize=(9, 2.5))
plt.plot(t, released)
plt.axvline(gate, linestyle="--", color="gray")
plt.title("released: only the part after the dashed line is used")
plt.xlabel("time (s)")
plt.show()

**拼起来:** 松开之前用 `held`,之后用 `released`。`np.where` 第 2 集见过。

In [ ]:
env = np.where(t < gate, held, released)

plt.figure(figsize=(9, 2.5))
plt.plot(t, env)
plt.axvline(gate, linestyle="--", color="gray")
plt.title("ADSR  (dashed line = key released)")
plt.xlabel("time (s)")
plt.show()

### 包成函数

In [ ]:
def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # 时间不能是 0,否则下面会除以 0
    # 按住期间:(0,0) → (a,1) → (a+d,s),之后停在 s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # 松开那一刻 envelope 有多高(可能还没走完 attack)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # 从那个高度用 r 秒直线降到 0;np.clip 让它降到 0 后不再变负
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # 松开前用 held,松开后用 released

## 5. 三种 articulation

**1. pluck** — attack 极短,sustain 为 0。

**应该听到:** 短促的 "咚",约 0.3 秒就没了。gate 有 1.2 秒,但声音早就结束了 — 因为 sustain 是 0。

In [ ]:
pluck = adsr(t, a=0.005, d=0.3, s=0.0, r=0.1, gate=1.2)
play(osc * pluck)

**2. pad** — attack 很长。

**应该听到:** 声音用 0.8 秒慢慢淡入,1.2 秒松开后再用 0.7 秒慢慢消失。

In [ ]:
pad = adsr(t, a=0.8, d=0.2, s=0.8, r=0.7, gate=1.2)
play(osc * pad)

**3. organ** — attack 和 release 都只有 5 毫秒,sustain 为 1。

**应该听到:** 立刻满音量,1.2 秒时立刻停,而且两头都没有 click。

In [ ]:
organ = adsr(t, a=0.005, d=0.01, s=1.0, r=0.005, gate=1.2)
play(osc * organ)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, e, name in zip(axes, [pluck, pad, organ], ["pluck", "pad", "organ"]):
    ax.plot(t, osc * e, linewidth=0.3)
    ax.plot(t, e, color="red")
    ax.set_ylabel(name)
axes[-1].set_xlabel("time (s)")
plt.show()

## 6. Gate 的长短

同一组 ADSR,只改 `gate`。

**应该听到:** 第一个是短音(按 0.15 秒),第二个是长音(按 1.4 秒)。音头的感觉完全一样,只是 "按住" 的时间不同。

注意第一个:0.15 秒时 attack(0.05)已经走完,decay(0.2)才走到一半,所以松开时的高度在 1 和 sustain 之间 — 这就是函数里要算 `level_at_release` 的原因。

In [ ]:
short_note = adsr(t, a=0.05, d=0.2, s=0.6, r=0.3, gate=0.15)
long_note = adsr(t, a=0.05, d=0.2, s=0.6, r=0.3, gate=1.4)

plt.figure(figsize=(9, 2.5))
plt.plot(t, short_note, label="gate 0.15")
plt.plot(t, long_note, label="gate 1.4")
plt.legend()
plt.xlabel("time (s)")
plt.show()

display(play(osc * short_note))
display(play(osc * long_note))

## 7. 曲线的形状

我们的 envelope 是直线段。但耳朵对音量的感觉不是线性的:直线下降听起来像 "先撑着,最后突然没了"。

真实的 synth 用的是 exponential 曲线:一开始掉得快,越来越慢。一个简单的近似:把 envelope **取三次方**。0 还是 0,1 还是 1,中间被压低。

**应该听到:** 两个 pluck。第一个(直线)比较 "硬";第二个(曲线)开头更脆,尾巴更自然,像真的拨弦。

In [ ]:
linear = adsr(t, a=0.005, d=0.8, s=0.0, r=0.1, gate=1.2)
curved = linear ** 3

plt.figure(figsize=(9, 2.5))
plt.plot(t, linear, label="linear")
plt.plot(t, curved, label="linear ** 3")
plt.legend()
plt.xlabel("time (s)")
plt.show()

display(play(osc * linear))
display(play(osc * curved))

## 8. 一段 bassline

把 "oscillator + envelope" 包成一个函数 `note()`,每调用一次就产生一个完整的音。然后像第 1 集做旋律那样,用 `np.concatenate` 接起来。

`gate=dur * 0.7`:每个音按住它时值的 70%,剩下 30% 留给 release。

循环里:

- `notes = []`:一个空 list
- `notes.append(...)`:往 list 末尾加一个东西
- 循环结束后 list 里有 8 个 array,`np.concatenate` 把它们接成一条

**应该听到:** 一段 8 个音的低音 pluck bassline,每个音 0.25 秒,共 2 秒。

In [ ]:
def note(freq, dur, a, d, s, r, pos=0.6):
    t = np.arange(int(sr * dur)) / sr
    phase = (freq * t) % 1
    env = adsr(t, a, d, s, r, gate=dur * 0.7)      # 按住 70%,留 30% 给 release
    return osc_wavetable(frames, phase, pos) * env ** 2

pitches = [55.00, 55.00, 82.41, 55.00, 65.41, 55.00, 98.00, 82.41]      # A1 A1 E2 A1 C2 A1 G2 E2

notes = []
for f in pitches:
    notes.append(note(f, 0.25, a=0.003, d=0.15, s=0.2, r=0.05))

bassline = np.concatenate(notes)

plt.figure(figsize=(9, 2.5))
plt.plot(np.arange(len(bassline)) / sr, bassline, linewidth=0.3)
plt.xlabel("time (s)")
plt.show()

play(bassline)

## 常见错误

| 现象 | 原因 |
|---|---|
| 音头或音尾有 "咔" | attack 或 release 是 0;或者 release 比留给它的时间长,被砍断了 |
| 改了 sustain 但听不出区别 | gate 比 attack + decay 短,还没走到 sustain 就松开了 |
| 把 sustain 当成时间填了 `2.0` | sustain 是高度(0–1),不是秒 |
| 声音完全没有 | envelope 全是 0。检查 `a`、`d`、`s` 是否写反 |

## 练习

**1.** 做一个 "反向" 的感觉:attack 1 秒,sustain 1,release 0.01,gate 1.0。**先猜:** 听起来像什么?

<details><summary>答案</summary>

```python
play(osc * adsr(t, a=1.0, d=0.01, s=1.0, r=0.01, gate=1.0))
```
声音慢慢涨起来然后突然停 — 像一段倒放的 pluck。
</details>

**2.** 把 bassline 的每个音改成 0.125 秒,并把 `pitches` 重复两遍。提示:`pitches + pitches` 会把 list 接起来。

<details><summary>答案</summary>

```python
notes = []
for f in pitches + pitches:
    notes.append(note(f, 0.125, a=0.003, d=0.08, s=0.2, r=0.03))
play(np.concatenate(notes))
```
</details>

**3.** 用 `np.interp` 做一条 "两个峰" 的 envelope:0 → 1 → 0.2 → 1 → 0,盖在 `osc` 上。

<details><summary>答案</summary>

```python
env2 = np.interp(t, [0, 0.05, 0.5, 0.55, 1.5], [0, 1, 0.2, 1, 0])
play(osc * env2)
```
</details>

## 小结

| Max | Python |
|---|---|
| `adsr~` | `adsr(t, a, d, s, r, gate)` |
| note on / note off | `gate`(按住多少秒) |
| `function` + `line~` | `np.interp(t, times, levels)` |
| `*~` | `osc * env` |
| `sig~ 0`(静音) | `np.zeros(n)` |

- envelope 是一个 array
- attack 和 release 至少几毫秒,否则有 click
- 第 4 集说过:array 可以接到任何参数上。同一个 `adsr()` 之后还会去控制 filter 和 wavetable position。

**下一集:** `lores~` — filter,以及 Python 里第一次必须写 sample 循环的地方。